# WSP on Colab

Select a hosted GPU runtime. The download cell prepares the public Sketchy v1 dataset in the current VM.

In [ ]:
from pathlib import Path
import subprocess

CONTENT = Path("/content")
if not CONTENT.is_dir():
    raise RuntimeError("Select a hosted Colab runtime.")
REPO = CONTENT / "WSP"
if not REPO.exists():
    subprocess.run(["git", "clone", "--depth", "1", "https://github.com/HieuNM1804/WSP.git", str(REPO)], check=True)
subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv"], check=True)

In [ ]:
import sys

SKETCHY = CONTENT / "Sketchy"
subprocess.run([sys.executable, str(REPO / "wsp_sbir/download_sketchy.py"), "--destination", str(SKETCHY)], check=True)

## Sandbox check

This check must succeed before launching the WSP driver. The editor executes its Python cells inside this Linux sandbox.

In [ ]:
import shutil

if not shutil.which("bwrap"):
    subprocess.run(["apt-get", "update", "-qq"], check=True)
    subprocess.run(["apt-get", "install", "-y", "-qq", "bubblewrap"], check=True)
subprocess.run(["bwrap", "--unshare-all", "--ro-bind", "/usr", "/usr",
                "--symlink", "usr/lib", "/lib", "--symlink", "usr/lib64", "/lib64",
                "/usr/bin/true"], check=True)

## Publish a result

Set the run directory below. The cell previews the metrics JSON. Once Git is authenticated with write access to the repository, set `publish = True` to push a `results/<run>` branch.

In [ ]:
run = REPO / "wsp_sbir/runs/s2_astra_50"
publish = False
command = [sys.executable, str(REPO / "wsp_sbir/publish_results.py"), "--run", str(run)]
if publish:
    command.append("--push")
subprocess.run(command, cwd=REPO, check=True)